# Machine Learning Workflow — Guided Examples

This notebook demonstrates a leakage-aware workflow with a locally generated synthetic dataset. Run cells from top to bottom. The data are illustrative and should not be treated as real customer evidence.

**Workflow:** frame → inspect → split → baseline → pipeline → evaluate → document.

## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports complete. Random seed:", RANDOM_STATE)

## 2. Frame the problem
We will demonstrate a binary classification task: predict whether a fictional customer will churn. The model score is a decision aid, not a causal explanation.

In [ ]:
X_raw, y = make_classification(
    n_samples=1500,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    weights=[0.78, 0.22],
    class_sep=0.9,
    random_state=RANDOM_STATE
)
df = pd.DataFrame(X_raw, columns=[f"numeric_{i}" for i in range(X_raw.shape[1])])
df["region"] = np.random.choice(["North", "South", "East", "West"], size=len(df))
df["plan"] = np.random.choice(["Basic", "Plus", "Premium"], size=len(df), p=[0.5, 0.32, 0.18])
df["churn"] = y
# Add a few missing values to make preprocessing meaningful.
for col in ["numeric_1", "numeric_4", "region"]:
    idx = np.random.choice(df.index, size=35, replace=False)
    df.loc[idx, col] = np.nan

df.head()

## 3. Inspect data quality

In [ ]:
print("Shape:", df.shape)
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())
print("\nTarget proportions:")
display(df["churn"].value_counts(normalize=True).rename("proportion").to_frame())

In [ ]:
df.select_dtypes(include="number").hist(figsize=(10, 6), bins=25)
plt.suptitle("Numeric feature distributions")
plt.tight_layout()
plt.show()

## 4. Define features and split
The target is `churn`; all other columns are candidate features. A stratified split keeps class proportions similar across train and test. For time-dependent data, use a chronological split instead.

In [ ]:
X = df.drop(columns="churn")
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", round(y_train.mean(), 3))
print("Test positive rate:", round(y_test.mean(), 3))

## 5. Establish a baseline
A majority-class baseline gives a simple reference. Accuracy can be deceptive when the target is imbalanced, so compare additional metrics.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print(classification_report(y_test, baseline_pred, zero_division=0))

## 6. Build preprocessing and model pipelines
Numeric columns are median-imputed and standardized. Categorical columns are imputed with their most frequent value and one-hot encoded. Since preprocessing is inside the pipeline, it is fitted only on training data during `fit` and inside each cross-validation fold.

In [ ]:
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)
print("Model fitted.")

## 7. Evaluate on the held-out test set
Use the test set only after model choices are made. Metrics below describe this synthetic example and do not guarantee production performance.

In [ ]:
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred, zero_division=0),
    "f1": f1_score(y_test, pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, proba),
}
pd.Series(metrics, name="test_metric").to_frame()

In [ ]:
print(classification_report(y_test, pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Confusion matrix — test set")
plt.show()

## 8. Compare baseline and candidate on the same test set

In [ ]:
def evaluate_predictions(y_true, y_pred, y_score=None):
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is not None:
        result["roc_auc"] = roc_auc_score(y_true, y_score)
    return result

comparison = pd.DataFrame({
    "majority_baseline": evaluate_predictions(y_test, baseline_pred),
    "logistic_regression": evaluate_predictions(y_test, pred, proba),
}).T
comparison

## 9. Cross-validation for model-selection evidence
Cross-validation uses only the training split here. It can provide a more stable estimate than a single validation split, but the split strategy must match the data-generating process.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    model, X_train, y_train, cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    return_train_score=False
)
pd.DataFrame({
    metric.replace("test_", ""): [cv_results[metric].mean(), cv_results[metric].std()]
    for metric in cv_results if metric.startswith("test_")
}, index=["mean", "std"]).T

## 10. Document the experiment
A useful record includes the purpose, data, split, model, metrics, limitations, and monitoring plan.

In [ ]:
experiment_record = {
    "problem": "Illustrative binary customer-churn classification",
    "data": "Synthetic data generated with sklearn.make_classification",
    "random_state": RANDOM_STATE,
    "split": "80/20 stratified train/test split",
    "preprocessing": "Median imputation + scaling for numeric; most-frequent imputation + one-hot encoding for categorical",
    "model": "LogisticRegression(max_iter=1000)",
    "test_metrics": metrics,
    "limitations": [
        "Synthetic data may not reflect real-world patterns.",
        "No temporal or group effects are represented.",
        "Metrics do not establish causal impact or fairness."
    ],
    "monitoring": [
        "Track input schema, missingness, and feature drift.",
        "Track prediction rates and delayed outcome metrics.",
        "Review performance across relevant groups where appropriate."
    ],
}
experiment_record

## 11. Optional extension
Try a different classifier, change the threshold using validation data, or use a chronological/group split if appropriate. Keep the test set untouched while selecting the model. Explain why your chosen metric matches the decision.

## Key takeaways
- Define the problem and evaluation criteria before modeling.
- Split before fitting learned transformations.
- Put preprocessing in a pipeline to reduce leakage risk.
- Compare candidates with a baseline and appropriate metrics.
- Keep experiment records and monitor the system after deployment.